# 06 — DINOv3 BYOT status (and optional embedding)

DINOv3 is gated (custom DINOv3 License — NOT Apache like DINOv2).

> VisionServeX does **not** redistribute gated or restricted model weights. You bring your own token and accept upstream licenses yourself. Tokens are always redacted.

In [ ]:
# Install the published package from PyPI (run AFTER release).
# Before release you may instead `pip install dist/visionservex-3.8.0-py3-none-any.whl`.
# %pip install -q visionservex==3.8.0
import importlib.metadata as _m
print("installed:", _m.version("visionservex"))

In [ ]:
# Assert we are using the INSTALLED package (site-packages), never the local src.
import visionservex
print("visionservex:", visionservex.__version__)
print("file:", visionservex.__file__)
assert "site-packages" in visionservex.__file__, (
    "This tutorial must run against the pip-installed package, not local src. "
    "Use a fresh venv / clean kernel and install visionservex from PyPI."
)

In [ ]:
from visionservex import VSX
acc = VSX.model("dinov3-vits16").access()
print("state:", acc.get("state"))

In [ ]:
# Optional real embedding — only if access granted.
emb = None
if acc.get("state") == "access_granted":
    from PIL import Image
    import numpy as np
    img = Image.fromarray((np.random.rand(224,224,3)*255).astype("uint8"))
    emb = VSX.dino("dinov3-vits16").embed(img)
    print("embedding_dim:", emb.get("embedding_dim"))
else:
    print("auth_required — accept at https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m")

In [ ]:
# Record an execution-ledger row (artifact or honest blocker).
import csv, json, os, time
from pathlib import Path
ART = Path("v38_tutorial_artifacts"); ART.mkdir(exist_ok=True)
def record(notebook, status, detail, artifact=None):
    art = None
    if artifact is not None:
        art = ART / f"{notebook}.json"
        art.write_text(json.dumps(artifact, indent=2, default=str))
    row = {"notebook": notebook, "status": status, "detail": detail,
           "artifact": str(art) if art else "", "version": __import__("visionservex").__version__}
    led = ART / "v38_tutorial_execution_ledger.csv"
    new = not led.exists()
    with led.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)
    print("ledger +", row)

In [ ]:
record("06_dinov3", "benchmark_passed_byot" if emb else "auth_required",
       acc.get("state",""), {"state": acc.get("state"), "embedding_dim": (emb or {}).get("embedding_dim")})